# Accepted PRs — Exploration

**Goal:** Identify and characterise AI-generated pull requests from the AIDev dataset that were **created, merged, and approved by a human reviewer** without requiring any changes — i.e. the AI's contribution was accepted as-is.

---

## What this notebook does

### 1. Data loading
Five tables are pulled from the `hao-li/AIDev` HuggingFace dataset:
- `pull_request` — PR metadata (title, agent, repo, created/closed/merged timestamps)
- `pr_commits` — commits attached to each PR
- `pr_reviews` — formal review events (APPROVED, CHANGES_REQUESTED, DISMISSED, COMMENTED)
- `pr_review_comments_v2` — inline review comments tied to specific diff hunks
- `pr_timeline` — full event timeline per PR

### 2. Isolating accepted PRs
A PR is considered **accepted** if:
- `merged_at` is non-null (was successfully merged), AND
- Its `review_category` is `approved_only` — meaning it received at least one APPROVED review with no CHANGES_REQUESTED or DISMISSED events

### 3. Bot noise removal
PRs where every APPROVED review was submitted by a bot (`user_type != 'User'`) are excluded, ensuring the acceptance signal comes from a genuine human reviewer.

### 4. Supporting subsets
Two filtered DataFrames are built over the final `accepted_prs` set:
- `pr_reviews_sub` — reviews on these PRs only
- `pr_commits_sub` — commits on these PRs only

### 5. Commit count analysis
Commit counts are joined so that squashed PRs (1 commit) vs. multi-commit PRs can be distinguished.

### 6. Summary + spot-check
A summary cell prints total counts. A spot-check cell lets you inspect one PR end-to-end (reviews, commits).

---

## Next steps
- Add any additional filters (e.g. minimum commit count, specific agents, date range).
- Deep-dive analysis cells go below the spot-check cell.

In [51]:
import pandas as pd
from huggingface_hub import hf_hub_download

def load_table(filename):
    path = hf_hub_download(repo_id="hao-li/AIDev", filename=filename, repo_type="dataset")
    return pd.read_parquet(path)

prs      = load_table("pull_request.parquet")
commits  = load_table("pr_commits.parquet")
reviews  = load_table("pr_reviews.parquet")
rev_cmts = load_table("pr_review_comments_v2.parquet")
timeline = load_table("pr_timeline.parquet")
details  = load_table("pr_commit_details.parquet")

In [52]:
import sys
sys.path.insert(0, '..')
from helpers import build_pr_states, categorize

# All merged PRs
merged = prs[prs['merged_at'].notna()].copy()

pr_states = build_pr_states(reviews, merged['id'])
merged['review_category'] = merged['id'].apply(lambda pr_id: categorize(pr_id, pr_states))

print(f"Merged PRs (raw): {len(merged):,}")
merged['review_category'].value_counts()

Merged PRs (raw): 24,014


review_category
no_reviews                                       18082
approved_only                                     3444
commented_only                                    1806
changes_requested_then_approved                    386
changes_requested_no_approval                      129
dismissed_then_approved                            112
changes_requested_and_dismissed_then_approved       41
dismissed_no_approval                               14
Name: count, dtype: int64

In [53]:
# Keep only PRs that were approved with no changes requested or dismissed events
accepted_prs = merged[merged['review_category'] == 'approved_only'].copy()

print(f"Accepted PRs (approved_only, raw): {len(accepted_prs):,}")
accepted_prs['review_category'].value_counts()

Accepted PRs (approved_only, raw): 3,444


review_category
approved_only    3444
Name: count, dtype: int64

In [54]:
# Filter: Remove PRs where every APPROVED review was from a bot
human_approved_pr_ids = set(
    reviews[
        (reviews['state'] == 'APPROVED') &
        (reviews['user_type'] == 'User')
    ]['pr_id'].unique()
)

accepted_prs = accepted_prs[accepted_prs['id'].isin(human_approved_pr_ids)]
print(f"After removing bot-approved PRs: {len(accepted_prs):,}")

After removing bot-approved PRs: 3,434


In [55]:
pr_reviews_sub = reviews[
    reviews['pr_id'].isin(accepted_prs['id'])
][['pr_id', 'id', 'user', 'user_type', 'state', 'submitted_at']].copy()

pr_commits_sub = commits[
    commits['pr_id'].isin(accepted_prs['id'])
][['pr_id', 'sha', 'author', 'message']].copy()

print(f"Reviews (on accepted PRs): {len(pr_reviews_sub):,}")
print(f"Commits (on accepted PRs): {len(pr_commits_sub):,}")

Reviews (on accepted PRs): 11,390
Commits (on accepted PRs): 16,357


In [56]:
# Commit count per PR
commit_count_map = pr_commits_sub.groupby('pr_id')['sha'].count()
accepted_prs['commit_count'] = accepted_prs['id'].map(commit_count_map).fillna(0).astype(int)

print(f"Accepted PRs: {len(accepted_prs):,}")
print()
print(accepted_prs['commit_count'].describe())

Accepted PRs: 3,434

count    3434.000000
mean        4.763250
std         5.170288
min         1.000000
25%         2.000000
50%         3.000000
75%         6.000000
max        30.000000
Name: commit_count, dtype: float64


In [57]:
# Files changed per PR — count distinct filenames across all commits, then bucket
files_per_pr = (
    details[details['pr_id'].isin(accepted_prs['id'])]
    .groupby('pr_id')['filename']
    .nunique()
    .rename('n_files_changed')
)

accepted_prs = accepted_prs.copy()
accepted_prs['n_files_changed'] = accepted_prs['id'].map(files_per_pr).fillna(0).astype(int)

bins   = [0, 1, 4, 9, float('inf')]
labels = ['1', '2–4', '5–9', '10+']
accepted_prs['files_bucket'] = pd.cut(
    accepted_prs['n_files_changed'],
    bins=bins, labels=labels, right=True
)

bucket_counts = accepted_prs['files_bucket'].value_counts().reindex(labels)
bucket_pct    = (bucket_counts / len(accepted_prs) * 100).round(1)

summary = pd.DataFrame({'count': bucket_counts, 'pct': bucket_pct})
print("=== Files changed per PR (before filter) ===")
print(summary.to_string())
print(f"\nTotal PRs: {len(accepted_prs):,}")

# Keep only PRs with 2–9 files changed
before = len(accepted_prs)
accepted_prs = accepted_prs[
    (accepted_prs['n_files_changed'] >= 2) &
    (accepted_prs['n_files_changed'] <= 9)
].copy()

print(f"\nRemoved {before - len(accepted_prs):,} PRs (1 file or 10+ files changed)")
print(f"Remaining PRs (2–9 files): {len(accepted_prs):,}")

=== Files changed per PR (before filter) ===
              count   pct
files_bucket             
1               890  25.9
2–4            1019  29.7
5–9             484  14.1
10+            1041  30.3

Total PRs: 3,434

Removed 1,931 PRs (1 file or 10+ files changed)
Remaining PRs (2–9 files): 1,503


In [58]:
# Count human comments per PR (COMMENTED-state reviews + inline review comments from humans)

# Human COMMENTED-state reviews (each counts as 1 comment)
human_review_comments = (
    pr_reviews_sub[
        (pr_reviews_sub['state'] == 'COMMENTED') &
        (pr_reviews_sub['user_type'] == 'User')
    ]
    .groupby('pr_id')
    .size()
    .rename('n_human_review_comments')
)

# Inline comments from human reviewers on any review belonging to these PRs
human_review_ids = set(
    pr_reviews_sub[pr_reviews_sub['user_type'] == 'User']['id'].unique()
)
human_inline = (
    rev_cmts[rev_cmts['pull_request_review_id'].isin(human_review_ids)]
    .groupby('pull_request_review_id')
    .size()
    .rename('inline_count')
    .reset_index()
)

review_to_pr = pr_reviews_sub.set_index('id')['pr_id'].to_dict()
human_inline['pr_id'] = human_inline['pull_request_review_id'].map(review_to_pr)
human_inline_per_pr = human_inline.groupby('pr_id')['inline_count'].sum().rename('n_human_inline_comments')

# Total human comments per PR
accepted_prs['n_human_comments'] = (
    accepted_prs['id'].map(human_review_comments).fillna(0) +
    accepted_prs['id'].map(human_inline_per_pr).fillna(0)
).astype(int)

zero_comment = accepted_prs[accepted_prs['n_human_comments'] == 0].copy()

print(f"Accepted PRs:                    {len(accepted_prs):,}")
print(f"  0 human comments:              {len(zero_comment):,}")
print(f"  1 human comment:               {(accepted_prs['n_human_comments'] == 1).sum():,}")
print(f"  2+ human comments:             {(accepted_prs['n_human_comments'] >= 2).sum():,}")

Accepted PRs:                    1,503
  0 human comments:              994
  1 human comment:               39
  2+ human comments:             470


In [59]:
print("=== Accepted PRs Clean Subset ===")
print(f"Total PRs:                      {len(accepted_prs):,}")
print()
print(accepted_prs['review_category'].value_counts().to_string())
print()
print(f"Agent breakdown:")
print(accepted_prs['agent'].value_counts().to_string())

=== Accepted PRs Clean Subset ===
Total PRs:                      1,503

review_category
approved_only    1503

Agent breakdown:
agent
Copilot         568
Devin           488
OpenAI_Codex    309
Cursor          116
Claude_Code      22


In [62]:
# Spot-check: inspect a PR with exactly 1 human comment
one_comment = accepted_prs[accepted_prs['n_human_comments'] == 0]
sample_id = one_comment.sample(10, random_state=42)['id'].iloc[0]
pr_html_url = prs.loc[prs['id'] == sample_id, 'html_url'].iloc[0]

print("=== PR ===")
print(prs[prs['id'] == sample_id][['id', 'title', 'agent', 'repo_url', 'html_url', 'created_at', 'merged_at']].to_string(index=False))

print("\n=== Reviews ===")
print(
    pr_reviews_sub[pr_reviews_sub['pr_id'] == sample_id]
    .sort_values('submitted_at')[['state', 'user', 'user_type', 'submitted_at']]
    .to_string(index=False)
)

print("\n=== Commits ===")
commits_sample = pr_commits_sub[pr_commits_sub['pr_id'] == sample_id][['sha', 'author', 'message']].copy()
commits_sample['url'] = pr_html_url + '/commits/' + commits_sample['sha']
print(commits_sample.to_string(index=False))

# COMMENTED-state review bodies (top-level review comments)
sample_review_ids = set(pr_reviews_sub[pr_reviews_sub['pr_id'] == sample_id]['id'].unique())
commented_reviews = reviews[
    (reviews['pr_id'] == sample_id) &
    (reviews['state'] == 'COMMENTED') &
    reviews['body'].notna() &
    (reviews['body'].str.strip() != '')
][['user', 'user_type', 'submitted_at', 'body']].sort_values('submitted_at')

print("\n=== Review Comments (top-level) ===")
if commented_reviews.empty:
    print("  (none)")
else:
    for _, row in commented_reviews.iterrows():
        print(f"[{row['submitted_at']}] {row['user']} ({row['user_type']})")
        print(f"  {row['body']}\n")

# Inline comments on any review of this PR
inline = rev_cmts[
    rev_cmts['pull_request_review_id'].isin(sample_review_ids)
][['user', 'path', 'body', 'created_at']].sort_values('created_at')

print("=== Inline Review Comments ===")
if inline.empty:
    print("  (none)")
else:
    for _, row in inline.iterrows():
        print(f"[{row['created_at']}] {row['user']} on {row['path']}")
        print(f"  {row['body']}\n")

=== PR ===
        id                                                               title agent                                  repo_url                                  html_url           created_at            merged_at
2952353649 Refactor createKnowledgeSuggestionTask to reduce payload parameters Devin https://api.github.com/repos/liam-hq/liam https://github.com/liam-hq/liam/pull/1015 2025-03-27T10:10:41Z 2025-03-28T10:57:40Z

=== Reviews ===
   state     user user_type         submitted_at
APPROVED    MH4GF      User 2025-03-27T10:21:09Z
APPROVED junkisai      User 2025-03-28T10:28:39Z

=== Commits ===
                                     sha                    author                                                                                                                                              message                                                                                        url
893a397b8f4edcf1a012970d08f7ffa247b355e8 devin-ai-integration[bot] Refactor cre